# Проверка сохранённой модели на другом котле — legacy

Модель загружается из выбранной папки. Обучение и калибровка здесь не выполняются.
Веса, нормировка, пороги и правила тревог берутся из экспорта. Excel используется
для сопоставления датчиков получателя с входами модели; он не меняет её состав.

1. Задайте MODEL_DIR, TARGET_UNIT, DATA_PATH. Для другого котла тип должен совпадать:
   GROUPS/GROUP_NAME можно задать вручную или использовать сохранённую группу.
2. Выберите период и запустите сверху вниз. Предыстория используется для прогрева,
   в итоговые таблицы попадает только выбранный период [PERIOD_START, PERIOD_END).
3. Смотрите sensor_audit, сводку, тревоги и графики. Файлы создаются в отдельной папке
   saved_model_checks, исходный экспорт не изменяется.

Для строгой проверки на незнакомом котле включите REQUIRE_UNSEEN_UNIT=True.
ALLOW_BEFORE_CUTOFF=True разрешает прогон по прошлому; такой результат помечается
как временное наложение и не является проверкой прогноза на будущем.
Сводка показывает наблюдаемую частоту/длительность тревог. Метрики обнаружения
аварий и ложных тревог по заявкам здесь не вычисляются.


## 1. Параметры


In [ ]:
# Выбирайте полный экспорт: папка с model.py и config.json (final либо месяц).
MODEL_DIR = None                  # например r'D:/models/тип/К-5/analogs/2025-04'
TARGET_UNIT = 'К-7'
DATA_PATH = None                  # parquet выбранного получателя; TimeValue либо DatetimeIndex
SENSOR_FILE = 'sensor_roles.xlsx'  # сопоставление имён получателя; None — только общие ключи/схема экспорта
GROUPS = {}                      # при необходимости {'БКЗ-500': ['К-5', 'К-6', 'К-7']}
GROUP_NAME = None                 # None — единственная подходящая группа
PERIOD_START = None               # включительно; None — начало датасета
PERIOD_END = None                 # не включительно; None — конец датасета
ALLOW_BEFORE_CUTOFF = True        # разрешить исторический перенос; отчёт покажет наложение по времени
REQUIRE_UNSEEN_UNIT = False       # True — запретить получателя, входившего в обучение
ALLOW_MISSING_TARGETS = False     # условия/нагрузка/состояния обязательны всегда
WARMUP_HOURS = None               # None — вся доступная предыстория; например 168 для ограничения
VIEW = None                      # None — top5, если он есть, иначе первый вид
OUTPUT_DIR = 'saved_model_checks'
MODEL_SEARCH_ROOT = 'models_legacy'
THREADS = 2

# Необязательные настройки JSON для run_overnight.py --config ...
from pathlib import Path
import os, sys, json
_parent = Path(globals().get('__file__', '03_Проверка_сохранённой_модели.ipynb')).resolve().parent
_candidates = [_parent, Path.cwd() / 'legacy', Path.cwd().parent / 'legacy']
PROJECT_DIR = next((p.resolve() for p in _candidates if (p/'lib/saved_model_runner.py').is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Откройте ноутбук из папки legacy рядом с lib.')
os.chdir(PROJECT_DIR)
for _path in [PROJECT_DIR, PROJECT_DIR/'lib']:
    if str(_path) in sys.path: sys.path.remove(str(_path))
    sys.path.insert(0,str(_path))
_settings = os.environ.get('SAVED_MODEL_CONFIG_JSON') or os.environ.get('BOILER_CONFIG_JSON')
if _settings:
    _values = json.loads(Path(_settings).read_text(encoding='utf8'))
    _unknown = set(_values) - {k for k in globals() if k.isupper()}
    if _unknown: raise ValueError(f'Неизвестные параметры: {sorted(_unknown)}')
    globals().update(_values)


## 2. Доступные модели


In [ ]:
import importlib
import saved_model_runner as S
S = importlib.reload(S)
from IPython.display import display
catalog = S.discover_models(MODEL_SEARCH_ROOT)
display(catalog)
if not MODEL_DIR or not DATA_PATH:
    raise ValueError('Укажите MODEL_DIR и DATA_PATH в первой ячейке. Пути из каталога можно скопировать.')


## 3. Прогон без переобучения


In [ ]:
RESULT = S.run(MODEL_DIR, DATA_PATH, TARGET_UNIT, flavor='legacy',
    sensor_file=SENSOR_FILE, groups=GROUPS, group_name=GROUP_NAME,
    period_start=PERIOD_START, period_end=PERIOD_END,
    allow_before_cutoff=ALLOW_BEFORE_CUTOFF, require_unseen_unit=REQUIRE_UNSEEN_UNIT,
    allow_missing_targets=ALLOW_MISSING_TARGETS, warmup_hours=WARMUP_HOURS,
    output_dir=OUTPUT_DIR, threads=THREADS)
print('Результаты:', RESULT['root'])
display(RESULT['summary'])


## 4. Группа, граница обучения и датчики


In [ ]:
meta = RESULT['metadata']
display({k: meta[k] for k in ['target_unit','sources','group','recipient_in_training',
    'cutoff','period_start','period_end_exclusive','evaluation_before_cutoff',
    'input_start','recipient_verified_by_headers','model_files_unchanged']})
display(RESULT['audit'])
if meta['recipient_in_training']:
    print('Получатель входил в обучение: это проверка применения групповой модели.')
else:
    print('Получатель не входил в обучение: проверяется перенос между котлами.')
if meta['evaluation_before_cutoff']:
    print('В периоде есть время до cutoff: это исторический прогон, а не проверка на будущем.')


## 5. Отношение к порогу и ошибки датчиков


In [ ]:
figure = S.plot(RESULT, VIEW)
display(figure)


## 6. Тревоги и сохранённые файлы


In [ ]:
display(RESULT['alarms'])
print('timeline.csv / timeline.parquet — ряд показателей и тревог по видам.')
print('summary.csv — покрытие, длительность тревог и число уведомлений.')
print('alarms.csv — эпизоды, включая начавшиеся до выбранного периода.')
print('sensor_errors_XX.parquet — ошибки датчиков по порядку views из run.json.')
print('sensor_audit.csv — соответствие входов; run.json — параметры и контрольные суммы модели.')
print('overview.png — график; папка:', RESULT['root'])
